# Auto Reel Generator — runs entirely free, no local install

Topic -> Gemini script -> Cloud TTS voice -> hosted SadTalker avatar (via Hugging Face) -> HyperFrames edit -> final MP4.

**Before running:** Runtime menu is fine on CPU — you don't need a GPU here, since the avatar rendering happens on Hugging Face's free servers.

You'll need:
1. A free Gemini API key: https://aistudio.google.com/apikey
2. (Optional, for Kannada/Tamil/etc voice) A Google Cloud project with the Text-to-Speech API enabled + a service account JSON key, OR skip this and use Gemini's own TTS for supported languages.
3. A photo of yourself uploaded in this notebook.

In [ ]:
!git clone https://github.com/YOUR-USERNAME/reel-generator.git
# ^ push this project to your own GitHub repo first, then clone it here.
# Or: skip the clone and just !pip install the packages + paste the .py files
# into notebook cells directly if you don't want to use GitHub.
%cd reel-generator
!pip install -q -r requirements.txt

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")
# If using Google Cloud TTS for Kannada/Tamil/etc, also upload your
# service-account.json via the Colab file browser (left sidebar) and set:
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "service-account.json"

In [ ]:
from google.colab import files
print("Upload a clear, front-facing photo of yourself:")
uploaded = files.upload()
photo_name = list(uploaded.keys())[0]
import shutil
shutil.move(photo_name, "avatar_engine/my_avatar.jpg")
print("Saved to avatar_engine/my_avatar.jpg")

In [ ]:
# Edit your topic + language here before running
import config
config.TOPIC = "5 morning habits that boost productivity"
config.LANGUAGE = "kn-IN"       # Kannada
config.NUM_SCENES = 5
config.AVATAR_BACKEND = "hosted"  # free, uses Hugging Face's GPU, no setup

In [ ]:
# Step 1: script
import script_generator
script = script_generator.generate_script()
script

In [ ]:
# Step 2: voice
# If Gemini native TTS doesn't support your language well (e.g. Kannada),
# swap this for the Google Cloud TTS version -- see voice_generator.py notes.
import voice_generator
audio_paths = voice_generator.generate_all_voices(script)
audio_paths

In [ ]:
# Step 3: avatar clips (hosted -- runs on Hugging Face's free GPU, may queue)
import avatar_generator
clip_paths = avatar_generator.generate_all_avatar_clips(audio_paths)
clip_paths

In [ ]:
# Step 4: build the HyperFrames scene
import hyperframes_builder
scene_path = hyperframes_builder.build_scene(script, clip_paths)
scene_path

In [ ]:
# Step 5: render with HyperFrames (needs Node.js -- Colab has it, or install below)
!npm install -g @heygen/hyperframes  # check the exact package name is current
!hyperframes render {scene_path} --out output/final_reel.mp4 --fps 30 --aspect 9:16

In [ ]:
from google.colab import files
files.download("output/final_reel.mp4")